# Sample Inference — One Race, One Prediction

Demonstrates the full pre-race inference path described in [`pipeline/inference_pipeline.md`](../inference_pipeline.md): given one race's driver/team lineup and everything knowable before lights-out, produce a predicted finishing order — end to end, using a persisted model instead of an in-notebook fit/eval loop like the other `pipeline/model/*.ipynb` notebooks.\n\n**Target race for this demo**: chosen as the most recently *completed* race with full data on disk as of today, rather than a fixed one, so re-running this notebook later naturally advances to the next finished round. Its actual `RacePosition` is masked to NaN before features are assembled (see `build_inference_features()`), so the model only ever sees what would genuinely have been known pre-race — the real result is only unmasked at the end, to sanity-check the prediction.\n\n**Known limitation carried over from the design doc**: weather is not in `FINAL_FEATURES` (no forecast pipeline exists yet), and imputation medians for lag/rolling features aren't persisted alongside the label encoders/scaler — a driver with no usable prior-race history (debut, or a DNF-lap-1 with no lap time) can't be transformed and is reported, not silently dropped."

## 1. Setup

In [8]:
import json
import sys
from pathlib import Path

import joblib
import pandas as pd
from lightgbm import LGBMRegressor

sys.path.insert(0, str(Path.cwd().parent.parent))
from pipeline.feature_engineering.feature_engineering import (
    FINAL_FEATURES,
    NUMERIC_SCALE_FEATURES,
    TARGET,
    build_inference_features,
    load_preprocessors,
)

DATA_ROOT = Path.cwd().parent.parent / "data"
MODEL_PATH = Path.cwd() / "lightgbm_model.joblib"
BEST_PARAMS_PATH = Path.cwd().parent.parent / "reports" / "lightgbm_best_params.json"

preprocessors = load_preprocessors()

## 2. Pick the Target Race

No live/upcoming-race feed exists yet (see the design doc's weather-forecast gap), so this demo stands in a real, already-ingested race for an unplayed one: the most recently *completed* race with a full `session=R` partition (`session_info`/`driver_info`/`session_results`/`laps`) on disk. A handful of the newest 2026 rounds only have an entry-list `session_results.parquet` (no lap data yet, since they haven't been run) and are skipped.

This has to happen *before* the model is fit (Section 3) — the training set otherwise includes the target race's own real result, since the existing train/test split only holds out year 2025, not "everything from the target race onward."

In [9]:
REQUIRED_FILES = ["session_info.parquet", "driver_info.parquet", "session_results.parquet", "laps.parquet"]

def find_latest_completed_race(data_root: Path) -> tuple[int, int]:
    candidates = []
    for year_dir in data_root.glob("year=*"):
        year = int(year_dir.name.split("=")[1])
        for round_dir in year_dir.glob("round=*"):
            r_dir = round_dir / "session=R"
            if r_dir.exists() and all((r_dir / f).exists() for f in REQUIRED_FILES):
                candidates.append((year, int(round_dir.name.split("=")[1])))
    if not candidates:
        raise FileNotFoundError(f"No complete session=R partitions found under {data_root}")
    return max(candidates)

target_year, target_round = find_latest_completed_race(DATA_ROOT)

session_info = pd.read_parquet(DATA_ROOT / f"year={target_year}" / f"round={target_round:02d}" / "session=R" / "session_info.parquet")
meeting_name = session_info["Meeting.Name"].iloc[0]
print(f"Target race: {target_year} round {target_round} - {meeting_name}")

Target race: 2026 round 11 - Hungarian Grand Prix


## 3. Load the Persisted Model

No model notebook in this repo calls `joblib.dump` — this is the gap flagged in `pipeline/inference_pipeline.md`'s [Known gaps](../inference_pipeline.md#known-gaps) ("No persisted model artifact"). This section closes it: if `lightgbm_model.joblib` doesn't already exist next to this notebook, fit the Optuna-tuned LightGBM (`reports/lightgbm_best_params.json`, the same configuration `reports/error_analysis.ipynb` already treats as "the best model by test MAE") and persist it here. Re-running this cell later just loads the saved model — it doesn't refit.

**Leakage guard**: `train.parquet`'s year/test split only holds out 2025 — every other season, including the rest of 2026, is "train." Since the target race chosen in Section 2 is itself 2026 data, it would otherwise sit in the training set with its own real result. Its row (and, for correctness, anything from the same season at or after it, in case a non-latest race is ever chosen instead) is excluded before fitting, so the model never sees the answer to the race it's about to predict.

In [10]:
if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
    print(f"Loaded persisted model from {MODEL_PATH}")
else:
    train = pd.read_parquet(DATA_ROOT / "train.parquet")

    # round_number is standardised in train.parquet by the same saved scaler
    # used for inference features, so translate target_round into that scaled
    # space to identify (and exclude) the target race's own training rows.
    round_idx = NUMERIC_SCALE_FEATURES.index("round_number")
    scaler = preprocessors["scaler"]
    scaled_target_round = (target_round - scaler.mean_[round_idx]) / scaler.scale_[round_idx]

    leak_mask = (train["year"] == target_year) & (train["round_number"] >= scaled_target_round - 1e-9)
    print(f"Excluding {leak_mask.sum()} row(s) from training - same season, at or after the target race")
    train = train[~leak_mask].reset_index(drop=True)

    with open(BEST_PARAMS_PATH) as fh:
        tuned_params = json.load(fh)["params"]

    model = LGBMRegressor(random_state=42, verbose=-1, **tuned_params)
    model.fit(train[FINAL_FEATURES], train[TARGET])

    joblib.dump(model, MODEL_PATH)
    print(f"Trained on {len(train)} rows and saved model -> {MODEL_PATH}")

Loaded persisted model from e:\f1-pit-wall\pipeline\model\lightgbm_model.joblib


## 4. Assemble Pre-Race Features

`build_inference_features()` (added alongside this notebook in `feature_engineering.py`) mirrors `build_features()`'s join and lag/rolling computation, but:
- masks the target race's own `RacePosition` to NaN *before* computing lag/rolling features, so this historical race behaves like a genuinely unplayed one
- keeps the target race's rows instead of dropping them for a NaN target (`build_features` would drop exactly the rows we need)
- transforms categoricals/numerics with the saved `LabelEncoder`/`StandardScaler` from Section 1 (`.transform`, never refit)

`strict=False` prints a warning instead of raising if a driver has no persisted median to impute a NaN feature with (e.g. a debut, or a driver whose *previous* race ended on lap 1 with no lap time to compute `LapStd_lag1` from), and leaves that cell NaN rather than dropping the driver. LightGBM handles NaN splits natively, so this doesn't block a prediction for that driver.

In [11]:
inference_features = build_inference_features(
    target_year, target_round, DATA_ROOT, preprocessors=preprocessors, strict=False
)

driver_info = pd.read_parquet(DATA_ROOT / f"year={target_year}" / f"round={target_round:02d}" / "session=R" / "driver_info.parquet")
inference_features = inference_features.merge(
    driver_info[["DriverId", "FullName", "TeamName"]].rename(columns={"TeamName": "TeamName_raw"}),
    on="DriverId", how="left",
)

print(f"{len(inference_features)} drivers assembled for {meeting_name} {target_year}")
inference_features[["FullName", "TeamName_raw"] + FINAL_FEATURES]

[features] skipping incomplete partition e:\f1-pit-wall\data\year=2026\round=12\session=R (missing ['session_info.parquet', 'driver_info.parquet', 'laps.parquet'])
[features] skipping incomplete partition e:\f1-pit-wall\data\year=2026\round=13\session=R (missing ['session_info.parquet', 'driver_info.parquet', 'laps.parquet'])
[features] skipping incomplete partition e:\f1-pit-wall\data\year=2026\round=14\session=R (missing ['session_info.parquet', 'driver_info.parquet', 'laps.parquet'])
[features] skipping incomplete partition e:\f1-pit-wall\data\year=2026\round=15\session=R (missing ['session_info.parquet', 'driver_info.parquet', 'laps.parquet'])
[features] skipping incomplete partition e:\f1-pit-wall\data\year=2026\round=16\session=R (missing ['session_info.parquet', 'driver_info.parquet', 'laps.parquet'])
[features] skipping incomplete partition e:\f1-pit-wall\data\year=2026\round=17\session=R (missing ['session_info.parquet', 'driver_info.parquet', 'laps.parquet'])
[features] skipp

,FullName,TeamName_raw,GridPosition,round_number,TeamName,Meeting.Circuit.ShortName,DriverFinish_lag1,DriverFinish_ewm,TeamFinish_ewm,DriverFinish_roll3_inseason,TeamFinish_roll3_inseason,LapStd_lag1
0,Alexander Albon,Williams,1.482022,-0.010185,13,5,0.778287,1.462811,1.446605,1.553235,1.733955,-0.120132
1,Fernando Alonso,Aston Martin,0.969346,-0.010185,2,5,1.472050,1.702576,2.032758,1.699062,2.019276,-0.054364
2,Kimi Antonelli,Mercedes,-0.568685,-0.010185,9,5,-1.649883,-1.055826,-0.638511,-0.925827,-0.793175,-0.231141
3,Arvid Lindblad,Racing Bulls,-0.226900,-0.010185,10,5,-0.262358,-0.255630,-0.283959,-0.415432,-0.426334,-0.160026
4,Oliver Bearman,Haas F1 Team,1.140238,-0.010185,6,5,0.604846,0.804463,0.932794,0.605358,0.918752,-0.049207
5,Gabriel Bortoleto,Audi,0.627561,-0.010185,3,5,-0.435798,-0.203022,0.540598,-0.342518,0.429630,-0.169280
6,Valtteri Bottas,Cadillac,1.823807,-0.010185,4,5,1.298609,1.875531,1.945417,1.771976,1.978516,0.032214
7,Franco Colapinto,Alpine,0.456669,-0.010185,1,5,-0.088917,0.065339,-0.005686,0.167877,0.185069,-0.161432
8,Pierre Gasly,Alpine,0.285777,-0.010185,1,5,0.084524,-0.061098,-0.005686,0.167877,0.185069,-0.127118
9,Isack Hadjar,Red Bull Racing,-0.397793,-0.010185,11,5,-0.782680,-0.951247,-0.809163,-1.071654,-0.874696,0.021434


## 5. Predict

The model outputs a continuous score per driver, not a valid 1..N permutation — `PredictedRank` (its sort order) is what should be read as "predicted finishing order," while `PredictedPosition` is the raw regression value.

In [12]:
predictions = inference_features[["DriverNumber", "FullName", "TeamName_raw"]].copy()
predictions["PredictedPosition"] = model.predict(inference_features[FINAL_FEATURES])
predictions = predictions.sort_values("PredictedPosition").reset_index(drop=True)
predictions["PredictedRank"] = predictions.index + 1

predictions.drop(columns="DriverNumber")

,FullName,TeamName_raw,PredictedPosition,PredictedRank
0,Charles Leclerc,Ferrari,4.648806,1
1,Lando Norris,McLaren,5.655997,2
2,Max Verstappen,Red Bull Racing,6.346607,3
3,Lewis Hamilton,Ferrari,6.405623,4
4,Oscar Piastri,McLaren,6.415418,5
5,George Russell,Mercedes,7.505562,6
6,Kimi Antonelli,Mercedes,7.713728,7
7,Isack Hadjar,Red Bull Racing,8.301671,8
8,Arvid Lindblad,Racing Bulls,10.247633,9
9,Liam Lawson,Racing Bulls,11.516405,10


## 6. The Headline Prediction

One race in, one prediction out: the model's pick for the race win.

In [13]:
winner = predictions.iloc[0]
print(f"Predicted winner - {meeting_name} {target_year}: {winner['FullName']} ({winner['TeamName_raw']})")

Predicted winner - Hungarian Grand Prix 2026: Charles Leclerc (Ferrari)


## 7. Sanity Check Against the Actual Result

Only possible here because the demo target is a historical race with a known outcome — a genuinely future race has nothing to check against. `RacePosition` was masked during feature assembly (Section 4), so this is the first place the real result is read.

In [14]:
session_results = pd.read_parquet(DATA_ROOT / f"year={target_year}" / f"round={target_round:02d}" / "session=R" / "session_results.parquet")
actual = session_results[["DriverNumber", "Position"]].rename(columns={"Position": "ActualPosition"})

comparison = predictions.merge(actual, on="DriverNumber", how="left").drop(columns="DriverNumber")
comparison = comparison.sort_values("PredictedRank").reset_index(drop=True)

mae = (comparison["PredictedPosition"] - comparison["ActualPosition"]).abs().mean()
print(f"MAE for this single race: {mae:.2f} (not a model evaluation - n=1 race, just a demo sanity check)")
comparison

MAE for this single race: 3.12 (not a model evaluation - n=1 race, just a demo sanity check)


,FullName,TeamName_raw,PredictedPosition,PredictedRank,ActualPosition
0,Charles Leclerc,Ferrari,4.648806,1,4.0
1,Lando Norris,McLaren,5.655997,2,1.0
2,Max Verstappen,Red Bull Racing,6.346607,3,2.0
3,Lewis Hamilton,Ferrari,6.405623,4,5.0
4,Oscar Piastri,McLaren,6.415418,5,20.0
5,George Russell,Mercedes,7.505562,6,7.0
6,Kimi Antonelli,Mercedes,7.713728,7,3.0
7,Isack Hadjar,Red Bull Racing,8.301671,8,6.0
8,Arvid Lindblad,Racing Bulls,10.247633,9,10.0
9,Liam Lawson,Racing Bulls,11.516405,10,8.0
